In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('RecipeNLG_dataset.csv')

In [2]:
df.head(5)

# Show full columns
pd.set_option('display.max_colwidth', None)

In [3]:
df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER'],
      dtype='object')

In [39]:
import re

# Keep only the necessary columns
df = df[['title', 'ingredients', 'directions']].dropna()

# Clean the title casing
def fix_title_case(text):
    if isinstance(text, str):
        fixed = text.title()
        return re.sub(r"'S\b", "'s", fixed)
    return text

df['title'] = df['title'].apply(fix_title_case)

# Clean ingredients and directions (bracket/quote fix)
def clean_list_column(text):
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return ", ".join(item.strip().strip('"\'') for item in items)
    except:
        pass
    return str(text).strip("[]\"'")

df['ingredients_clean'] = df['ingredients'].apply(clean_list_column)
df['directions_clean'] = df['directions'].apply(clean_list_column)

# Show full columns
pd.set_option('display.max_colwidth', None)

# Preview the cleaned result
df[['title', 'ingredients_clean', 'directions_clean']].head(3)


,title,ingredients_clean,directions_clean
0,No-Bake Nut Cookies,"1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits","In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine., Stir over medium heat until mixture bubbles all over top., Boil and stir 5 minutes more. Take off heat., Stir in vanilla and cereal; mix well., Using 2 teaspoons, drop and shape into 30 clusters on wax paper., Let stand until firm, about 30 minutes."
1,Jewell Ball's Chicken,"1 small jar chipped beef, cut up, 4 boned chicken breasts, 1 can cream of mushroom soup, 1 carton sour cream","Place chipped beef on bottom of baking dish., Place chicken on top of beef., Mix soup and cream together; pour over chicken. Bake, uncovered, at 275° for 3 hours."
2,Creamy Corn,"2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. cream cheese, cubed, 1/3 c. butter, cubed, 1/2 tsp. garlic powder, 1/2 tsp. salt, 1/4 tsp. pepper","In a slow cooker, combine all ingredients. Cover and cook on low for 4 hours or until heated through and cheese is melted. Stir well before serving. Yields 6 servings."


In [40]:
# Vegetarian Filter

# Define a list of common non-vegetarian ingredients
non_veg_keywords = ['chicken', 'beef', 'pork', 'fish', 'shrimp', 'bacon', 'turkey', 'lamb', 'ham', 'sausage']

# Create vegetarian flag
def is_vegetarian(ingredients):
    return not any(meat in ingredients for meat in non_veg_keywords)

df['vegetarian'] = df['ingredients_clean'].apply(is_vegetarian)

In [41]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['ingredients_clean'])

# Build recommender function
def recommend_recipes(user_ingredients, top_n=5, vegetarian_only=False):
    user_vec = vectorizer.transform([user_ingredients])
    similarity = cosine_similarity(user_vec, tfidf_matrix)
    top_indices = similarity[0].argsort()[-top_n:][::-1]
    
    # Get top matching recipes
    results = df.iloc[top_indices]

    if vegetarian_only:
        results = results[results['vegetarian']]

    return results[['title', 'ingredients_clean', 'directions_clean']].head(top_n)

In [42]:
# Show recommendations in a clean way
import re
import ast

def show_recommendations(results):
    for i, row in results.iterrows():
        print(f"\n🍽️ {row['title']}")
        print(f"🧂 Ingredients: {row['ingredients_clean']}")
        print("📖 Directions:")

        try:
            steps = ast.literal_eval(row['directions_clean'])
            if isinstance(steps, list):
                for step in steps:
                    for sentence in step.split("."):
                        # Cleaning each sentence here
                        sentence = sentence.strip()
                        sentence = re.sub(r'^[\s\(\)\{\}\.,:;!?\"\'-]+', '', sentence)
                        if sentence:
                            print(f"- {sentence}.")
            else:
                raise ValueError("Not a list")
        except:
            for sentence in str(row['directions_clean']).split("."):
                sentence = sentence.strip()
                sentence = re.sub(r'^[\s\(\)\{\}\.,:;!?\"\'-]+', '', sentence)
                if sentence:
                    print(f"- {sentence}.")
# Example input
input_ingredients = "evaporated milk, brown sugar, peanut butter"

# Get recommendations
recommendations = recommend_recipes(input_ingredients)

In [43]:
show_recommendations(recommendations)


🍽️ Peanut Butter Sundae Topping
🧂 Ingredients: 1 c. sugar, 1 c. milk (evaporated), 1/2 c. peanut butter
📖 Directions:
- Cook until thick, about 10 minutes.
- Add a chunk of butter.
- Serve over ice cream for a delicious treat.

🍽️ Peanut Butter Fudge
🧂 Ingredients: 1 c. sugar, 2 Tbsp. butter, 1 tsp. vanilla, 1/4 lb. peanut butter, 1 c. brown sugar, 1/2 c. evaporated milk
📖 Directions:
- Cook sugar, butter, milk and salt to soft ball stage.
- Add peanut butter just before removing from fire.
- Do not stir!, Cool to room temperature.
- Add vanilla and beat until mixture is creamy, thick and will hold its shape.
- Pour on greased dish.

🍽️ *Peanut Butter Fudge
🧂 Ingredients: 1 1/2 c. peanut butter, 3 c. sugar, 1 c. evaporated milk, 1 tsp. vanilla
📖 Directions:
- Mix sugar and milk in heavy, deep pot.
- Let mixture come to a boil.
- Cook over low heat until it forms a soft ball when tested in cold water.
- Remove.
- Add peanut butter and vanilla.
- Beat until creamy.
- Pour into greased d

In [30]:
df_trimmed = df.head(20000)
df_trimmed.to_csv("cleaned.csv", index=False)